In [1]:
!pip install -q openai

In [2]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

print("Groq API connected successfully!")

Groq API connected successfully!


In [3]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": "Say hello!"}
    ]
)

print(response.choices[0].message.content)

Hello! 🌟 How can I help you today?


## Task 1: Prompt Parts


In [10]:
SYSTEM_PROMPT = """
You are a helpful Grocery Store Assistant.

FACTS:
- We sell rice, wheat, vegetables, fruits, and groceries.
- We deliver within our local service area.

RULES:
- Never promise delivery without checking availability.
- Never invent delivery timings.
- Ask the customer for their location before confirming delivery.
"""

print(SYSTEM_PROMPT)


You are a helpful Grocery Store Assistant.

FACTS:
- We sell rice, wheat, vegetables, fruits, and groceries.
- We deliver within our local service area.

RULES:
- Never promise delivery without checking availability.
- Never invent delivery timings.
- Ask the customer for their location before confirming delivery.



In [11]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Can you deliver in 1 hour?"}
    ]
)

print(response.choices[0].message.content)

I’d be happy to check on that for you! Could you let me know your delivery address or zip code so we can confirm availability and timing?


### Observation

Without RULES:
The AI invented a delivery estimate of 2–4 hours, even though this information was not provided.

With RULES:
The AI avoided promising a delivery time and asked for the customer's delivery address to confirm availability and timing.

Conclusion:
Adding rules to the system prompt helps the AI provide more reliable and controlled responses.

## Task 2: Few-Shot Learning

Objective: Build a Grocery Store message classifier using three examples and test it with five new messages.

In [12]:
examples = [
    {"message": "Do you have basmati rice?", "category": "Product Availability"},
    {"message": "What is the price of wheat?", "category": "Price Inquiry"},
    {"message": "Can you deliver groceries to my home?", "category": "Delivery Inquiry"}
]

for example in examples:
    print("Message:", example["message"])
    print("Category:", example["category"])
    print()

Message: Do you have basmati rice?
Category: Product Availability

Message: What is the price of wheat?
Category: Price Inquiry

Message: Can you deliver groceries to my home?
Category: Delivery Inquiry



In [13]:
FEW_SHOT_PROMPT = """
You are a Grocery Store Message Classifier.

Classify each customer message into exactly one of these categories:
- Product Availability
- Price Inquiry
- Delivery Inquiry

Here are some examples:

Message: Do you have basmati rice?
Category: Product Availability

Message: What is the price of wheat?
Category: Price Inquiry

Message: Can you deliver groceries to my home?
Category: Delivery Inquiry

Return only the category name.
"""

print(FEW_SHOT_PROMPT)


You are a Grocery Store Message Classifier.

Classify each customer message into exactly one of these categories:
- Product Availability
- Price Inquiry
- Delivery Inquiry

Here are some examples:

Message: Do you have basmati rice?
Category: Product Availability

Message: What is the price of wheat?
Category: Price Inquiry

Message: Can you deliver groceries to my home?
Category: Delivery Inquiry

Return only the category name.



In [14]:
test_messages = [
    "Are apples available today?",
    "How much does 1 kg of sugar cost?",
    "Do you deliver to Kakinada?",
    "Is sunflower oil available?",
    "Can I get home delivery for vegetables?"
]

for message in test_messages:
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": FEW_SHOT_PROMPT},
            {"role": "user", "content": message}
        ]
    )

    category = response.choices[0].message.content.strip()

    print("Message:", message)
    print("Category:", category)
    print("-" * 40)

Message: Are apples available today?
Category: Product Availability
----------------------------------------
Message: How much does 1 kg of sugar cost?
Category: Price Inquiry
----------------------------------------
Message: Do you deliver to Kakinada?
Category: Delivery Inquiry
----------------------------------------
Message: Is sunflower oil available?
Category: Product Availability
----------------------------------------
Message: Can I get home delivery for vegetables?
Category: Delivery Inquiry
----------------------------------------


### Observation

The Grocery Store classifier successfully classified all five new customer messages into the correct categories.

- Product Availability: 2 messages
- Price Inquiry: 1 message
- Delivery Inquiry: 2 messages

Accuracy: 5/5 = 100%

Conclusion:
Few-shot learning helps an AI understand a classification task by providing a small number of examples in the system prompt.

## Task 3: JSON Output

Objective: Extract customer order details in JSON format and parse the response using json.loads() inside try/except.

In [16]:
import json

order_prompt = """
Extract the grocery order details from the customer message.

Return only valid JSON with these exact fields:
- name
- item
- qty
- date
- eggless

If a field is not provided, use null.
"""

customer_message = """
My name is kavitha. I want 2 kg of basmati rice
for delivery on 30 November 2026.
"""

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "system", "content": order_prompt},
        {"role": "user", "content": customer_message}
    ]
)

print(response.choices[0].message.content)

```json
{
  "name": "kavitha",
  "item": "basmati rice",
  "qty": 2,
  "date": "30 November 2026",
  "eggless": null
}
```


In [19]:
import re

try:
    raw_response = response.choices[0].message.content

    # Remove Markdown code fences
    clean_response = re.sub(r"```json|```", "", raw_response).strip()

    # Convert JSON text into Python dictionary
    order_data = json.loads(clean_response)

    print("Order Details:")
    print("Name:", order_data["name"])
    print("Item:", order_data["item"])
    print("Quantity:", order_data["qty"])
    print("Date:", order_data["date"])
    print("Eggless:", order_data["eggless"])

except json.JSONDecodeError:
    print("Error: Invalid JSON response")

except Exception as e:
    print("Error:", e)

Order Details:
Name: kavitha
Item: basmati rice
Quantity: 2
Date: 30 November 2026
Eggless: None


### Observation

The AI successfully extracted the grocery order details and returned them in JSON format.

The json.loads() function converted the JSON response into a Python dictionary.

Initially, parsing failed because the AI included Markdown code fences. We removed them using Python's re module and successfully parsed the JSON.

Results:
- Name: kavitha
- Item: basmati rice
- Quantity: 2
- Date: 30 November 2026
- Eggless: None

Conclusion:
JSON helps store structured order information, and try/except handles parsing errors safely.

## Task 4: Memory

Objective: Understand how conversation memory works by setting MAX_TURNS = 2 and testing memory across three turns.

In [28]:
MAX_TURNS = 2

conversation_history = []

print("Maximum conversation turns:", MAX_TURNS)

Maximum conversation turns: 2


In [29]:
def chat_with_memory(user_message):
    global conversation_history

    # Add the new user message
    conversation_history.append(
        {"role": "user", "content": user_message}
    )

    # Keep only the latest MAX_TURNS completed turns
    conversation_history = conversation_history[-(MAX_TURNS * 2):]

    # Send the limited conversation history to the AI
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful Grocery Store Assistant. Answer only using the conversation context."
            }
        ] + conversation_history
    )

    assistant_reply = response.choices[0].message.content

    # Store the AI response
    conversation_history.append(
        {"role": "assistant", "content": assistant_reply}
    )

    print("AI:", assistant_reply)

In [30]:
chat_with_memory("My name is Kavitha.")

AI: Nice to meet you, Kavitha! How can I help you today?


In [31]:
chat_with_memory("What is the price of basmati rice?")

AI: I’m sorry, but I don’t have that information.


In [32]:
chat_with_memory("Do you deliver groceries to my home?")

AI: I’m sorry, but I don’t have that information.


In [33]:
chat_with_memory("What is my name?")

AI: I’m sorry, but I don’t have that information.


In [34]:
print("Conversation History:")
for message in conversation_history:
    print(message["role"], ":", message["content"])

Conversation History:
assistant : I’m sorry, but I don’t have that information.
user : Do you deliver groceries to my home?
assistant : I’m sorry, but I don’t have that information.
user : What is my name?
assistant : I’m sorry, but I don’t have that information.


### Observation

We set MAX_TURNS = 2 and introduced ourselves in the first conversation.

After chatting for three turns, we asked the AI to recall our name.

The AI could not remember the name because the first conversation was removed from the limited conversation history.

Conclusion:
Conversation memory helps an AI retain recent messages. Limiting the number of turns means older information may be forgotten.

## Task 5: My Business – Grocery Store Assistant

In [35]:
grocery_catalogue = {
    "basmati rice": {"price": 120, "unit": "kg"},
    "wheat": {"price": 40, "unit": "kg"},
    "sugar": {"price": 45, "unit": "kg"},
    "apples": {"price": 100, "unit": "kg"},
    "sunflower oil": {"price": 150, "unit": "litre"}
}

print("Grocery Store Catalogue:")
for product, details in grocery_catalogue.items():
    print(product, ":", "₹", details["price"], "per", details["unit"])

Grocery Store Catalogue:
basmati rice : ₹ 120 per kg
wheat : ₹ 40 per kg
sugar : ₹ 45 per kg
apples : ₹ 100 per kg
sunflower oil : ₹ 150 per litre


In [36]:
def get_product_details(product_name):
    """
    Get the price and unit of a grocery product
    from the grocery store catalogue.
    """
    product_name = product_name.lower()

    if product_name in grocery_catalogue:
        return grocery_catalogue[product_name]
    else:
        return "Product not found in our catalogue."


# Test the function
print(get_product_details("basmati rice"))
print(get_product_details("sugar"))
print(get_product_details("milk"))

{'price': 120, 'unit': 'kg'}
{'price': 45, 'unit': 'kg'}
Product not found in our catalogue.


In [37]:
SYSTEM_PROMPT = """
You are a friendly and helpful Grocery Store Assistant.

BUSINESS:
We are a local grocery store.

PRODUCT CATALOGUE:
- Basmati rice: ₹120 per kg
- Wheat: ₹40 per kg
- Sugar: ₹45 per kg
- Apples: ₹100 per kg
- Sunflower oil: ₹150 per litre

RULES:
- Answer customer questions politely.
- Provide product prices only from the catalogue.
- Never invent product prices or availability.
- If a product is not listed, say that its details are unavailable.
- Ask for the customer's location before discussing delivery confirmation.
- Never promise a delivery time without confirmation.
"""

print(SYSTEM_PROMPT)


You are a friendly and helpful Grocery Store Assistant.

BUSINESS:
We are a local grocery store.

PRODUCT CATALOGUE:
- Basmati rice: ₹120 per kg
- Wheat: ₹40 per kg
- Sugar: ₹45 per kg
- Apples: ₹100 per kg
- Sunflower oil: ₹150 per litre

RULES:
- Answer customer questions politely.
- Provide product prices only from the catalogue.
- Never invent product prices or availability.
- If a product is not listed, say that its details are unavailable.
- Ask for the customer's location before discussing delivery confirmation.
- Never promise a delivery time without confirmation.



In [38]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "What is the price of basmati rice?"}
    ]
)

print("Grocery Store Assistant:")
print(response.choices[0].message.content)

Grocery Store Assistant:
The price of Basmati rice is ₹120 per kilogram.


In [39]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "What is the price of milk?"}
    ]
)

print("Grocery Store Assistant:")
print(response.choices[0].message.content)

Grocery Store Assistant:
I’m sorry, but milk is not listed in our product catalogue. If you have any other questions or need assistance with something else, feel free to let me know!


### Task 5 Observation

I created my own Grocery Store Assistant.

- Built a grocery product catalogue using a Python dictionary.
- Created a function with a docstring to retrieve product details.
- Designed a custom SYSTEM prompt with business information and rules.
- Tested the AI with a listed product and an unlisted product.

Results:
- Basmati rice: Correctly returned ₹120 per kg.
- Milk: Correctly informed the customer that it was not listed.

Conclusion:
A custom SYSTEM prompt and business catalogue help build an AI assistant that responds according to specific business requirements.